# Classical machine learning for speech emotion recognition (Tunisian dialect)

This notebook is the **first stage of the model comparison**: every audio clip is summarised by a vector of hand-crafted acoustic features (MFCC, chroma, mel spectrogram), and classical classifiers (SVM, decision tree, random forest) predict one of four emotions: `angry`, `happy`, `neutral`, `sad`.

**Pipeline**
1. Extract features from each `.wav` file (mean of each feature over time)
2. Look up the emotion label of each clip in the annotation file
3. Train SVMs, decision trees and random forests
4. Compare them on the held-out test clips

> **About the outputs.** The outputs stored in this notebook were saved by the original runs on Google Colab (2022). The cleaned code has not been re-run on the original audio (the corpus is not part of this repository). Small formatting differences between the code and the stored outputs come from that clean-up.

## 1. Setup and data layout

The code expects the audio to be split into a training and a test folder, plus one annotation file:

```
data/
├── annotations.xlsx        # columns: path (file name, e.g. 15-3.wav), label (angry / happy / neutral / sad)
├── train/<video>/<video>-<n>.wav
└── val/<video>/<video>-<n>.wav     # used as the test set
```

Each clip is a short segment cut from a longer recording, named `<video>-<segment>.wav`.

The 80/20 split was created with the `split-folders` package (`splitfolders.ratio(source, output="data", seed=1337, ratio=(.8, .2))`). It splits **each video folder separately**, so segments of the same video can appear in both the training and the test folder. See the README for what this means for the results.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")                       # edit: folder that holds train/, val/ and the annotation file
ANNOTATION_FILE = DATA_DIR / "annotations.xlsx"
TRAIN_GLOB, TEST_GLOB = "train/**/*.wav", "val/**/*.wav"

# Which acoustic features to use (the original runs compared several combinations)
FEATURES = dict(mfcc=True, chroma=True, mel=True)   # 40 + 12 + 128 = 180 values per clip

EMOTIONS = ["angry", "happy", "neutral", "sad"]

In [ ]:
import numpy as np
import pandas as pd
import librosa
import soundfile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Feature extraction

For each clip, every feature is computed frame by frame and then **averaged over time**, which gives a fixed-length vector whatever the duration of the clip:

| Feature | Values per clip | What it captures |
|---|---|---|
| MFCC | 40 | the overall spectral shape of the voice (timbre) |
| Chroma | 12 | the energy in each of the 12 pitch classes |
| Mel spectrogram | 128 | the energy in perceptual frequency bands |

In [ ]:
def extract_feature(file_name, mfcc=False, chroma=False, mel=False):
    """Return one vector for a wav file: the time-average of the selected features."""
    with soundfile.SoundFile(str(file_name)) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate
    if X.ndim > 1:                       # stereo -> mono
        X = X.mean(axis=1)

    result = np.array([])
    if mfcc:
        mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
        result = np.hstack((result, mfccs))
    if chroma:
        stft = np.abs(librosa.stft(X))
        chroma_vec = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
        result = np.hstack((result, chroma_vec))
    if mel:
        mel_vec = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
        result = np.hstack((result, mel_vec))
    return result

## 3. Load the data

The annotation file maps each file name to its emotion. Clips whose label is not one of the four emotions are skipped, and clips without an annotation are counted and reported instead of being silently ignored.

In [ ]:
annotations = pd.read_excel(ANNOTATION_FILE)
label_of = dict(zip(annotations["path"], annotations["label"]))


def load_data(pattern):
    """Extract features and labels for every wav file matching `pattern` inside DATA_DIR."""
    X, y, missing = [], [], 0
    for file in sorted(DATA_DIR.glob(pattern)):
        label = label_of.get(file.name)
        if label is None:
            missing += 1
            continue
        if label not in EMOTIONS:
            continue
        X.append(extract_feature(file, **FEATURES))
        y.append(label)
    if missing:
        print(f"warning: {missing} file(s) without annotation were skipped")
    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = load_data(TRAIN_GLOB)
X_test, y_test = load_data(TEST_GLOB)
print(X_train.shape, X_test.shape)

(5587, 180)


The 5,587 training clips of the original run are the **augmented** training set: the 3,881 clips of the original (imbalanced) training set, plus `angry`, `happy` and `sad` clips from a second corpus built from YouTube videos, plus augmented copies of `angry` and `happy` clips (white noise, time stretching, pitch shifting; see notebook 01). The test set never contains these additions. The test set has 1,025 clips: 228 `angry`, 108 `happy`, 394 `neutral`, 295 `sad`. It is **not** balanced, so macro-F1 (the average of the per-class F1 scores) is a fairer metric than accuracy alone.

## 4. Models

`evaluate` trains a model, then prints the classification report, the accuracy and the confusion matrix on the test clips.

In [ ]:
def evaluate(model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print(classification_report(y_test, predictions))
    print("----accuracy score %s ----" % (accuracy_score(y_test, predictions) * 100))
    print(confusion_matrix(y_test, predictions, labels=EMOTIONS))
    return predictions


def plot_confusion(y_true, y_pred, title=""):
    cm = confusion_matrix(y_true, y_pred, labels=EMOTIONS)
    ax = sns.heatmap(pd.DataFrame(cm, EMOTIONS, EMOTIONS), cmap="BuGn", fmt="d", annot=True)
    ax.set_ylabel("True emotion")
    ax.set_xlabel("Predicted emotion")
    ax.set_title(title)
    plt.show()

### Support vector machines (linear, polynomial and RBF kernels)

In [ ]:
_ = evaluate(SVC(kernel='linear', C=1))

0.5736585365853658
              precision    recall  f1-score   support

       angry       0.56      0.55      0.55       228
       happy       0.33      0.18      0.23       108
     neutral       0.56      0.68      0.62       394
         sad       0.67      0.59      0.63       295

    accuracy                           0.57      1025
   macro avg       0.53      0.50      0.51      1025
weighted avg       0.57      0.57      0.56      1025

----accuracy score 57.36585365853658 ----
[[125  14  70  19]
 [ 24  19  55  10]
 [ 54  12 269  59]
 [ 21  13  86 175]]


In [ ]:
_ = evaluate(SVC(kernel='poly', degree=4))

0.5551219512195122
              precision    recall  f1-score   support

       angry       0.47      0.61      0.53       228
       happy       0.40      0.13      0.20       108
     neutral       0.57      0.65      0.61       394
         sad       0.66      0.55      0.60       295

    accuracy                           0.56      1025
   macro avg       0.52      0.48      0.48      1025
weighted avg       0.55      0.56      0.54      1025

----accuracy score 55.51219512195122 ----
[[138  10  69  11]
 [ 30  14  55   9]
 [ 66  10 256  62]
 [ 62   1  71 161]]


In [ ]:
_ = evaluate(SVC(kernel='rbf'))

0.5658536585365853
              precision    recall  f1-score   support

       angry       0.51      0.56      0.54       228
       happy       0.57      0.04      0.07       108
     neutral       0.53      0.75      0.62       394
         sad       0.73      0.51      0.60       295

    accuracy                           0.57      1025
   macro avg       0.59      0.47      0.46      1025
weighted avg       0.59      0.57      0.54      1025

----accuracy score 56.58536585365853 ----
[[128   1  90   9]
 [ 26   4  75   3]
 [ 51   2 297  44]
 [ 44   0 100 151]]


### Decision trees

In [ ]:
_ = evaluate(DecisionTreeClassifier(max_depth=4))

              precision    recall  f1-score   support

       angry       0.46      0.45      0.45       228
       happy       0.50      0.02      0.04       108
     neutral       0.49      0.72      0.58       394
         sad       0.66      0.49      0.56       295

    accuracy                           0.52      1025
   macro avg       0.53      0.42      0.41      1025
weighted avg       0.53      0.52      0.49      1025

----accuracy score 52.0 ----
[[102   0  97  29]
 [ 26   2  79   1]
 [ 62   2 284  46]
 [ 32   0 118 145]]


In [ ]:
_ = evaluate(DecisionTreeClassifier(max_depth=6))

              precision    recall  f1-score   support

       angry       0.50      0.39      0.44       228
       happy       0.00      0.00      0.00       108
     neutral       0.49      0.77      0.60       394
         sad       0.68      0.49      0.57       295

    accuracy                           0.53      1025
   macro avg       0.42      0.41      0.40      1025
weighted avg       0.50      0.53      0.49      1025

----accuracy score 52.58536585365854 ----
[[ 88   7 114  19]
 [ 23   0  83   2]
 [ 36   4 305  49]
 [ 28   5 116 146]]


### Random forests

In [ ]:
rf_predictions = evaluate(RandomForestClassifier(n_estimators=100, random_state=0))

0.6390243902439025
              precision    recall  f1-score   support

       angry       0.64      0.60      0.62       228
       happy       0.38      0.14      0.20       108
     neutral       0.59      0.73      0.65       394
         sad       0.75      0.73      0.74       295

    accuracy                           0.64      1025
   macro avg       0.59      0.55      0.55      1025
weighted avg       0.63      0.64      0.62      1025

[[136   5  78   9]
 [ 20  15  62  11]
 [ 45   9 288  52]
 [ 11  10  58 216]]


In [ ]:
_ = evaluate(RandomForestClassifier(n_estimators=20, random_state=0))

0.6136585365853658
              precision    recall  f1-score   support

       angry       0.62      0.62      0.62       228
       happy       0.38      0.23      0.29       108
     neutral       0.59      0.66      0.62       394
         sad       0.71      0.68      0.70       295

    accuracy                           0.61      1025
   macro avg       0.57      0.55      0.56      1025
weighted avg       0.61      0.61      0.61      1025

----accuracy score 61.36585365853659 ----
[[141  10  68   9]
 [ 19  25  52  12]
 [ 52  18 261  63]
 [ 17  13  63 202]]


In [ ]:
plot_confusion(y_test, rf_predictions, 'Random forest (100 trees)')

## 5. Results

All models below were evaluated on the same 1,025 test clips. The per-class F1 columns show where each model fails.

**MFCC + chroma + mel (180 features), 5,587 training clips**

| Model | Accuracy | Macro-F1 | Weighted-F1 | F1 angry | F1 happy | F1 neutral | F1 sad |
|---|---|---|---|---|---|---|---|
| SVM, linear | 0.57 | 0.51 | 0.56 | 0.55 | 0.23 | 0.62 | 0.63 |
| SVM, polynomial (degree 4) | 0.56 | 0.48 | 0.54 | 0.53 | 0.20 | 0.61 | 0.60 |
| SVM, RBF | 0.57 | 0.46 | 0.54 | 0.54 | 0.07 | 0.62 | 0.60 |
| Decision tree, depth 4 | 0.52 | 0.41 | 0.49 | 0.45 | 0.04 | 0.58 | 0.56 |
| Decision tree, depth 6 | 0.53 | 0.40 | 0.49 | 0.44 | 0.00 | 0.60 | 0.57 |
| **Random forest, 100 trees** | **0.64** | **0.55** | 0.62 | 0.62 | 0.20 | 0.65 | 0.74 |
| Random forest, 20 trees | 0.61 | 0.56 | 0.61 | 0.62 | 0.29 | 0.62 | 0.70 |

**Mel spectrogram only (128 features)** (a second, shorter run; the training-set size was not printed in that run)

| Model | Accuracy | Macro-F1 | Weighted-F1 |
|---|---|---|---|
| SVM, linear | 0.45 | 0.35 | 0.42 |
| SVM, RBF | 0.47 | 0.37 | 0.44 |
| SVM, polynomial (degree 4) | 0.42 | 0.25 | 0.33 |
| Decision tree, depth 4 | 0.52 | 0.41 | 0.50 |
| Decision tree, depth 6 | 0.51 | 0.39 | 0.48 |
| Random forest, 100 trees | 0.61 | 0.52 | 0.59 |
| Random forest, 20 trees | 0.59 | 0.52 | 0.58 |

**What to take from it**
- **Random forests are clearly the best classical models** (macro-F1 0.55 to 0.56), and the full 180 values beat mel alone (0.52).
- **MFCC alone is as good as all 180 values.** The thesis ran each feature separately (these runs are not in the recovered notebooks): with a random forest, MFCC alone gives accuracy 0.64 and weighted-F1 0.63, the same as the 180 values (0.64 and 0.62), while chroma alone is the weakest (accuracy 0.35) and mel alone reaches 0.61.
- **`happy` is the hard class** for every model (F1 between 0.00 and 0.29). It is the smallest class (108 test clips) and the one most often confused with `neutral`.
- The number of trees and the tree depth were compared on the test clips themselves, so the best values are slightly optimistic.
- **Which F1?** The "F1-score" columns of the thesis tables are the **weighted** F1 (the support-weighted average of the four per-class scores), which is higher than the macro-F1 because `happy`, the worst class, is also the smallest. The weighted-F1 column above matches the thesis numbers (for example random forest: 0.62).